# Extended Pupil Mapping, Aplanatism and Imaging Notes

**Optional companion to 5.4.** The main lesson now follows one lithography example from wavevector to angular sigma coordinates and, under a stated sine mapping, physical pupil positions. This notebook retains the earlier extended calculations: reference caps, aplanatic design examples, sampled illumination supports, coherent LSI transfer and mask/wafer conventions. Its independent examples use their own parameter sets.

Prerequisites: [5.2: coordinate representations](../5_2_Direction_Cosines_and_Spatial_Frequency_Coordinates.ipynb) and [5.3: two-dimensional order lattices](../5_3_Two_Dimensional_Gratings_and_Diffraction_Order_Lattices.ipynb). Coefficient derivations in 7.4 are a forward extension, not a prerequisite.

Run independently in order with NumPy and Matplotlib. Preserve $e^{-i\omega t}$ and outgoing $-z$. Use vacuum wavelength $\lambda_0$, $k_0=2\pi/\lambda_0$ and explicit lossless indices. $\lambda_{\rm out}=\lambda_0/n_{\rm out}$ is the outgoing in-medium wavelength. All grating lengths are in micrometres; physical pupil positions in the small calculation are in millimetres. The application terminology at the end is optional.


In [ ]:
from pathlib import Path
import sys, importlib
_support_dir = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                     if (p/'wave_optics'/'__init__.py').is_file()), None)
if _support_dir is None:
    raise FileNotFoundError('Open within the Grating_Physics folder.')
if str(_support_dir) not in sys.path:
    sys.path.insert(0, str(_support_dir))
import wave_optics
wave_optics.__path__[:] = [str(_support_dir/'wave_optics')]
from wave_optics import lattice_grating as lg
importlib.reload(lg)
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

from wave_optics.lattice_plotting import coordinate_plot


## 1. Angular acceptance comes after diffraction

An aligned circular objective in outgoing index $n_{\rm out}$ accepts propagating directions satisfying
$$\frac{|\mathbf k_\parallel|}{k_0}\le\mathrm{NA}_{\rm obj},\qquad 0<\mathrm{NA}_{\rm obj}\le n_{\rm out}.$$
The same boundary is
$$\alpha^2+\beta^2\le\left(\frac{\mathrm{NA}_{\rm obj}}{n_{\rm out}}\right)^2,
\qquad f_x^2+f_y^2\le\left(\frac{\mathrm{NA}_{\rm obj}}{\lambda_0}\right)^2.$$
Define generic normalized angular pupil coordinates
$$\boxed{\boldsymbol\sigma=\frac{\mathbf k_\parallel}{k_0\mathrm{NA}_{\rm obj}}
=\frac{n_{\rm out}}{\mathrm{NA}_{\rm obj}}(\alpha,\beta)
=\frac{\lambda_0}{\mathrm{NA}_{\rm obj}}(f_x,f_y)}.$$
Collection requires propagation toward the objective and $|\boldsymbol\sigma|\le1$. The grazing boundary is instead $|\boldsymbol\sigma|=n_{\rm out}/\mathrm{NA}_{\rm obj}$. Grazing orders never reach a downstream plane, even when the two circles coincide. Evanescent candidates have no real outgoing direction to collect. Neither changing NA nor normalizing coordinates changes the diffraction wavevectors.

Reuse the generic $p_x=4$, $p_y=5$, $\lambda_0=2$ grating in index 1. First calculate geometry; then apply an objective with NA 0.8. All uncollected labels are retained.


In [ ]:
wavelength0, px, py, NA = 2., 4., 5., .8
A = np.diag([px,py])
v = np.arange(-6,7)
mm,nn = np.meshgrid(v,v)
labels = np.column_stack([mm.ravel(),nn.ravel()])
geometry = lg.lattice_orders(A,labels,wavelength0=wavelength0)
data = lg.collect_orders(geometry,NA)


In [ ]:
def order_plot(ax, d, title, n_out=1., na=NA, annotate=False):
    sigma = d['sigma']
    styles = [(d['collected'], 'tab:blue', 'o', 'Propagating, collected'),
              ((d['status']=='propagating') & ~d['collected'], 'tab:orange', 'o', 'Propagating, excluded'),
              (d['status']=='evanescent', '.6', 'x', 'Evanescent'),
              (d['status']=='grazing', 'tab:purple', 'D', 'Grazing')]
    for keep, color, marker, label in styles:
        ax.scatter(*sigma[keep].T, color=color, marker=marker, s=22, label=label)
    ax.add_patch(Circle((0,0), 1, fill=False, color='tab:blue'))
    ax.add_patch(Circle((0,0), n_out/na, fill=False, color='black', linestyle='--'))
    if annotate:
        for mn, xy in zip(d['labels'], sigma):
            if max(abs(mn)) <= 1:
                ax.annotate(f'({int(mn[0])},{int(mn[1])})', xy, xytext=(4,4), textcoords='offset points', fontsize=8)
    ax.set(title=title, xlabel=r'$\sigma_x$', ylabel=r'$\sigma_y$', aspect='equal', xlim=(-4,4), ylim=(-4,4))
    ax.grid(alpha=.2)

fig, ax = plt.subplots(figsize=(7,6), constrained_layout=True)
order_plot(ax, data, 'Order lattice: unit collection circle; dashed propagation circle', annotate=True)
ax.legend(loc='upper left', fontsize=8)
plt.show()


## 2. From an order direction to a pupil position

Section 1 tells us which directions an objective accepts. To locate those orders in a **physical pupil plane**, we also need the optical mapping. For the ideal sine mapping used here,
$$r=C\sin\theta,\qquad R=C\sin\theta_{\max},$$
where $r$ is the distance from the pupil center, $R$ its outer radius, and $C$ a length scale set by the chosen optics. Dividing cancels that scale. With pupil axes aligned to the transverse direction coordinates,
$$\boxed{\left(\frac{x_p}{R},\frac{y_p}{R}\right)
=\frac{n}{\mathrm{NA}_{\rm obj}}(\alpha,\beta).}$$
Tran and Oldenbourg use this relation to calibrate pupil positions with known grating directions [7, Eqs. (1)–(2), Fig. 2](https://pmc.ncbi.nlm.nih.gov/articles/PMC8005462/).

This pupil follows a **sine law**, whereas the bare observation plane at $z=-L$ follows $r_D=L\tan\theta$. The optical system supplies the different mapping. Aplanatic imaging combines the sine condition with stigmatic on-axis imaging, correcting spherical aberration and coma in the ideal design [10](https://doi.org/10.1364/OE.25.010710).
- Here we use its geometrical mapping part at a specified pupil or relay; the object-to-image sine condition alone does not establish the mapping at every internal aperture [12](https://patents.google.com/patent/WO2005015316A2/en).


### Angular labels, pupil position and reference caps

For the declared sine mapping, pupil axes point along the same transverse axes as $\boldsymbol\sigma$, so $(x_p,y_p)/R_p=\boldsymbol\sigma$ with pupil radius $R_p$. The physical scale $R_p$ belongs to the chosen pupil or relay and is not fixed by NA alone. A reference cap has its own center and radius $R_c$: a real direction $\hat{\mathbf k}$ labels a point $\mathbf r_c=\mathbf c+R_c\hat{\mathbf k}$. Its radius is not the pupil radius.

A cap marker is a direction label, not the value of a field there. Evaluating a field on the cap requires the actual surface points and phases, for example $\sum C_{mn}e^{i\mathbf k_{mn}\cdot\mathbf r_c}$ when coefficients refer to the laboratory origin. Entrance and exit caps can label the corresponding directions on either side of an imaging system; the lens transformation and reference phases must be supplied separately. A bare observation plane follows a tangent projection, not the sine mapping.


### Small calculation: locate an order in normalized angular coordinates and check collection

Use the earlier real order with $(\alpha,\beta)=(0.8,0.4)$. Put the grating in a medium with $n=1.33$ and collect with an ideal objective of $\mathrm{NA}_{\rm obj}=1.2$. Take an illustrative pupil radius $R=3\,\mathrm{mm}$.

This calculation holds the outgoing direction fixed and applies the collection and mapping rules; it does not recompute diffraction for a changed medium at fixed vacuum wavelength. Physical pupil positions below are in **millimetres**.

**Predict and rerun:** reduce only the objective NA to $1.0$. Does the normalized pupil radius become larger or smaller? Does the order become evanescent, or does a real order simply fall outside the collection aperture?


In [ ]:
alpha, beta = 0.8, 0.4  # one outgoing order's direction cosines
n, NA_obj = 1.33, 1.2
R_mm = 3.0  # illustrative physical pupil radius

if (not np.all(np.isfinite([alpha, beta, n, NA_obj, R_mm]))
        or not 0 < NA_obj <= n or R_mm <= 0):
    raise ValueError("Use finite parameters, 0 < NA_obj <= n, and R_mm > 0.")

sin_theta_squared = alpha**2+beta**2
if sin_theta_squared > 1+1e-12:
    print("Evanescent candidate: there is no real outgoing direction to collect.")
elif abs(sin_theta_squared-1) <= 1e-12:
    print("Grazing candidate: no propagation toward an objective below the grating.")
else:
    na_coordinates = n*np.array([alpha, beta])
    radial_na = np.linalg.norm(na_coordinates)
    print("NA coordinates (n*alpha, n*beta) =", np.round(na_coordinates, 6))
    print("Radial NA coordinate =", round(radial_na, 6))
    sigma_coordinates = na_coordinates/NA_obj
    rho = np.linalg.norm(sigma_coordinates)
    print("sigma coordinates (sigma_x, sigma_y) =", np.round(sigma_coordinates, 6))
    print("Normalized pupil radius rho =", round(rho, 6), "(collection edge: 1)")
    collected = rho <= 1+1e-12
    print("Real outgoing order collected:", bool(collected))
    if collected:
        print("Sine-mapped pupil (xp/R, yp/R) =", np.round(sigma_coordinates, 6))
        print("Physical pupil (xp, yp), mm =", np.round(R_mm*sigma_coordinates, 6))
    else:
        print("The order still propagates, but it has no collected point inside this pupil.")


<details><summary>Answer</summary>

The NA coordinates are $(1.064,0.532)$, with radial coordinate approximately $1.1896$. With NA $1.2$, the q coordinates are $(0.8867,0.4433)$ and $\rho_m\approx0.9913<1$. The objective therefore collects this order near its pupil edge:
$$\left(\frac{x_p}{R},\frac{y_p}{R}\right)\approx(0.8867,0.4433),\qquad
(x_p,y_p)=(2.66,1.33)\,\mathrm{mm}.$$

Reducing the objective NA to $1.0$ gives q coordinates $(1.064,0.532)$ and $\rho_m\approx1.1896>1$, so the order is excluded. Its wavevector and propagation state have not changed; dividing by a smaller collection NA places it farther from the origin on the normalized scale. These q coordinates remain meaningful outside the collection disk. The physical pupil mapping used for the collected case assumes the sine condition.

</details>


<details><summary>Optional examples: other systems designed toward the sine condition</summary>


| Optical system | Concrete example and role of the condition |
| --- | --- |
| **Corrected immersion microscope objectives** | Tran and Oldenbourg use 60×/1.2 NA water-immersion and 60×/1.4 NA oil-immersion Plan Apo objectives, calibrating angle-to-aperture position using known grating orders [7, §2.1 and Table 2](https://pmc.ncbi.nlm.nih.gov/articles/PMC8005462/). |
| **Aplanatic high-NA condensers** | Farré and colleagues analyze a Nikon T-CHA oil-immersion condenser, NA 1.4, for optical-trap detection. Their ray tracing and measurements check the sine mapping needed to relate detector position to transverse optical momentum [8, Fig. 11](https://doi.org/10.1364/OE.20.012270). |
| **Schwarzschild-based aplanatic mirror systems** | Wolter–Schwarzschild telescope designs use mirror shapes constrained by the sine condition together with on-axis focusing. They illustrate deliberate aplanatic design using reflecting surfaces [9](https://openaccess.inaf.it/entities/publication/0c76045c-44a4-4277-8fc0-85e1c57d0f97). |

These are design conditions at specified wavelengths, conjugates and surrounding media. A manufactured system's mapping accuracy is something to verify under its actual operating conditions. The circular pupil assumed above is the simple introductory case; mirror systems can also have a central obscuration.


</details>


## 3. A sampled illumination pupil: keep source identity

For each sampled incident direction $s$, retain $(s,m,n)$ and calculate its own translated lattice. In normalized coordinates,
$$\boldsymbol\sigma_{s,mn}=\boldsymbol\sigma_s+\frac{m\mathbf b_1+n\mathbf b_2}{k_0\mathrm{NA}_{\rm obj}}.$$
An arbitrary sampled support can be supplied as an array of $\boldsymbol\sigma_s$; physical incident direction cosines are $(\mathrm{NA}_{\rm obj}/n_{\rm in})\boldsymbol\sigma_s$. Below, an annular source supplies a compact extended-illumination example. Each selected order translates the entire sampled source support, then the objective pupil clips it.

These dots encode **angular support**, not amplitude weighting. Source amplitudes, quadrature weights, and mutual coherence must be specified separately. Overlapping source-order distributions do not by themselves imply coherent interference. For mutually incoherent sources, intensities of each source's coherent order sum would be added; that imaging calculation belongs to subsequent work.


In [ ]:
axis = np.linspace(-.6,.6,19)
sx, sy = np.meshgrid(axis,axis)
source_sigma = np.column_stack([sx.ravel(), sy.ravel()])
radius = np.linalg.norm(source_sigma,axis=1)
source_sigma = source_sigma[(radius>=.3) & (radius<=.6)]
source_records = []
for source_id, qs in enumerate(source_sigma):
    d = lg.collect_orders(lg.lattice_orders(A, labels, wavelength0=wavelength0, incident=NA*qs), NA)
    source_records.append(dict(source_id=source_id, source_sigma=qs.copy(),
                               **d))
fig, axes = plt.subplots(1,2, figsize=(10,4.5), constrained_layout=True)
for pair, color in zip([(0,0),(1,0),(-1,0),(0,1),(1,1)],
                       ['black','tab:blue','tab:orange','tab:green','tab:purple']):
    j = np.flatnonzero(np.all(labels==pair,axis=1))[0]
    points = np.array([r['sigma'][j] for r in source_records])
    keep = np.array([r['collected'][j] for r in source_records])
    axes[0].scatter(*points.T, s=9, color=color, label=str(pair))
    axes[1].scatter(*points[keep].T, s=9, color=color)
for ax, title in zip(axes, ['Shifted source supports', 'After projection-pupil clipping']):
    ax.add_patch(Circle((0,0),1,fill=False,color='tab:blue'))
    ax.set(title=title, xlabel=r'$\sigma_x$', ylabel=r'$\sigma_y$', aspect='equal', xlim=(-1.6,1.6),ylim=(-1.1,1.7))
    ax.grid(alpha=.2)
axes[0].legend(title='Order (m,n)', fontsize=8)
plt.show()
print(len(source_records), 'source identities retained; no amplitudes assigned by support')


## 4. A brief coherent linear shift-invariant model

The grating supplies the input field to the **downstream imaging system**. Its periodic scattering or prescribed transmission is a separate model. Assume a monochromatic scalar field, fixed polarization, a stationary linear optical system, and an isoplanatic field region where a shift-invariant approximation holds. Choose input-referred image coordinates: for signed magnification $M$, compare $U_{\rm image}(M\mathbf r)$ to the input at $\mathbf r$, with any uniform gain included in the transfer normalization.

**Linearity** means $\mathcal H\{aU_1+bU_2\}=a\mathcal H\{U_1\}+b\mathcal H\{U_2\}$ for complex fields. **Shift invariance** means shifting the input by $\Delta\mathbf r$ shifts the response by the same displacement in these input-referred coordinates (by $M\Delta\mathbf r$ in physical image coordinates). Angular acceptance alone establishes neither this property nor the complete system transfer.

In this model each transverse spatial-frequency component is an eigenfunction: it acquires a complex amplitude and phase factor. The **coherent transfer function** $H(\mathbf f_\parallel)$ describes those factors. Thus each grating order undergoes
$$\boxed{C_{mn}\longrightarrow H(\mathbf f_{\parallel,mn})C_{mn}}.$$
For physical image coordinates the corresponding frequency scales as $\mathbf f'_{\parallel,mn}=\mathbf f_{\parallel,mn}/M$. The coefficients need specified reference planes and origins; known propagation phase may be included in $H$.

A specified complex pupil response $P(x_p,y_p)$ describes transmission magnitude and phase. Under an ideal coherent Fourier-imaging model with the declared sine mapping, a common reference convention, and a suitable pupil conjugate to the angular spectrum, one can write
$$H(\mathbf f_\parallel)=g\,P\!\left(R_p\frac{\lambda_0}{\mathrm{NA}_{\rm obj}}\mathbf f_\parallel\right),$$
where $g$ is a chosen uniform system gain and the pupil is zero outside the accepted aperture. Any required apodization, aberration or reference-phase factor must be included in the specified response. A hard circular acceptance supplies only the zero/nonzero support of $H$, not its full complex values. An arbitrary internal aperture is not automatically this Fourier pupil.

The **impulse response** $h$ is the complex field response to a point input in the same coordinate convention; it is the inverse transverse Fourier transform of $H$. Convolution derivations, PSFs, detailed lens propagation and partially coherent imaging are left to the later series.

### Small calculation: acceptance and complex transfer answer different questions

Compare orders $(0,0)$, $(1,0)$ and $(2,0)$ from the normal-incidence example. First read their pupil coordinates and acceptance. For illustration only, prescribe unit input coefficients and a pupil phase $\phi(\boldsymbol\sigma)=0.3|\boldsymbol\sigma|^2$ radians, with unit amplitude inside the aperture and zero outside. Then $H=\mathrm{acceptance}\,e^{i\phi}$.

**Predict:** can two accepted orders keep their magnitudes but acquire different phases? What happens to an excluded order? The phase law is chosen for this example; it is not a calculated lens response.


In [ ]:
for order in [(0, 0), (1, 0), (2, 0)]:
    j = np.flatnonzero(np.all(data['labels'] == order, axis=1))[0]
    sigma = data['sigma'][j]
    accepted = bool(data['collected'][j])
    phase = .3*np.dot(sigma, sigma)  # illustrative pupil phase, radians
    H = np.exp(1j*phase) if accepted else 0j
    C_in = 1.+0j  # prescribed example amplitude at the input reference plane
    C_out = H*C_in
    phase_text = f'{np.angle(C_out):.3f} rad' if abs(C_out) > 0 else 'undefined'
    print(f'{order}: |sigma|={np.linalg.norm(sigma):.3f}, collected={accepted}, '
          f'output magnitude={abs(C_out):.1f}, output phase={phase_text}')


## 5. Optional application: lithography

The preceding sections stand without these conventions. Here “mask” denotes the object and “wafer” the reduced image. For $\lambda_0=193$ nm, signed magnification $M=-1/4$ (inversion), and positive reduction $D=1/|M|=4$, wafer-side NA 1.35 corresponds under the ideal sine condition to mask-side NA $|M|1.35=0.3375$. The mask side is air. Wafer-side index must be at least 1.35; NA does not uniquely specify its immersion index.

A mask pitch of 800 nm maps geometrically to a wafer pitch of 200 nm. At normal incidence, mask-side order NA coordinates are $(0.24125m,0)$; dividing by **mask-side** NA gives $\boldsymbol\sigma_m\approx(0.7148m,0)$. Thus $0,\pm1$ are collected and $\pm2$ propagate outside the aperture. Do not mix wafer pitches with mask-side NA.

In this application the components of $\boldsymbol\sigma$ are conventionally written $(\sigma_x,\sigma_y)$. The distinct scalar source extent $\sigma=\mathrm{NA}_{\rm illum}/\mathrm{NA}_{\rm obj}$ describes a centered source disk on the mask-side normalized scale; it does not determine mutual coherence. For the declared aligned-axis mapping, wafer-side NA coordinates are $1/M=-4$ times the mask-side ones and normalized angular coordinates invert. A physical pupil radius of 3 mm would place the mask-side $+1$ order at about $x_p=2.144$ mm. Complex coefficients and system transfer are still required for an image.

The magnification convention and examples follow the references below ([11](#ref-11), [14](#ref-14), [15](#ref-15)).


## Final checkpoint and handoff

Can an order propagate but miss the objective? Does a direction label determine a pupil position in millimetres? Does a circular acceptance establish an LSI imaging model?

<details><summary>Answers</summary>

Yes: the collection disk is generally smaller than the propagation disk. Physical pupil position also requires the optical mapping and a length scale. No: shift invariance requires stated system and field-region assumptions; acceptance supplies only angular support.

</details>

For every source, retain source identity, order label, wavevector, propagation state and collection state. Attach complex coefficients at a stated reference plane when a transmission or scattering model supplies them. Preserve illumination weights and coherence separately. In the LSI approximation a declared $H$ then transforms those coefficients; entrance/exit reference caps label directions without replacing the field transformation.

[Previous: 5.3](../5_3_Two_Dimensional_Gratings_and_Diffraction_Order_Lattices.ipynb) · [Chapter 5](../5_0_Grating_Diffraction_in_Three_Dimensions.ipynb) · [Next: amplitude and intensity](../6_0_Amplitude_and_Intensity_of_a_Scalar_Wave.ipynb) · [Forward: complex coefficients](../7_4_Diffraction_Kinematics_and_Complex_Order_Coefficients.ipynb)


## References and conventions

<a id="ref-7"></a>

7. **Tran and Oldenbourg (2021; online 2020)**, [*An experimental method to characterize the relationship between aperture image and ray directions in microscope optics*](https://pmc.ncbi.nlm.nih.gov/articles/PMC8005462/), *Microscopy Research and Technique* **84**, 668–674; [DOI](https://doi.org/10.1002/jemt.23625). Equations (1)–(2), Figure 2, and Table 2 connect the sine mapping, NA and measured pupil coordinates.

<a id="ref-8"></a>

8. **Farré, Marsà and Montes-Usategui (2012)**, [*Optimized back-focal-plane interferometry directly measures forces of optically trapped particles*](https://doi.org/10.1364/OE.20.012270), *Optics Express* **20**, 12270–12291, especially Figure 11: sine-condition checks for the collecting condenser and optical system.

<a id="ref-9"></a>

9. **Sironi (2017)**, [*Aplanatic telescopes based on Schwarzschild optical configuration: from grazing incidence Wolter-like x-ray optics to Cherenkov two-mirror normal incidence telescopes*](https://openaccess.inaf.it/entities/publication/0c76045c-44a4-4277-8fc0-85e1c57d0f97), *Proceedings of SPIE* **10399**, 1039903; [DOI](https://doi.org/10.1117/12.2275422). Examples of mirror systems designed using the sine condition.

<a id="ref-10"></a>

10. **Narasimhan et al. (2017)**, [*Three surface freeform aplanatic systems*](https://doi.org/10.1364/OE.25.010710), *Optics Express* **25**, 10710–10715, §§1–2: aplanatism, stigmatic imaging, and the sine condition. Only those definitions are used here.

<a id="ref-11"></a>

11. **Peng, Hu, Tolani, Dam, Tyminski and Slonaker (2010)**, [*Toward a consistent and accurate approach to modeling projection optics*](https://doi.org/10.1117/12.848252), *Proceedings of SPIE* **7640**, 76402Y. [Author-uploaded manuscript](https://www.researchgate.net/publication/229050169_Toward_a_Consistent_and_Accurate_Approach_to_Modeling_Projection_Optics), Eqs. (27)–(30): component form of the sine condition and mask-to-wafer spatial-frequency scaling. Their positive demagnification factor $M$ is our reduction ratio $D$; this notebook reserves $M$ for signed transverse magnification. Their wavelength is vacuum wavelength.

<a id="ref-12"></a>

12. **Shafer, Epple, Ulrich and Beierl / Carl Zeiss SMT (2005)**, [*Projection objective for microlithography*, WO2005015316A2](https://patents.google.com/patent/WO2005015316A2/en), Summary of the Invention and the description of the radiance-transformation group. Primary design disclosure explicitly correcting the sine condition at the wafer; also distinguishes that correction from the mapping at an internal pupil.

<a id="ref-14"></a>

14. **Mack**, [*CHE323/384: Chemical Processes for Micro- and Nanofabrication, Homework 11 Solutions*](https://lithoguru.com/scientist/CHE323/Homework11_solutions.pdf), University of Texas at Austin, problem 3 (PDF page 2): sigma space as transverse spatial frequency normalized by the objective cutoff, with order step $\lambda_0/(d\,\mathrm{NA})$. The source writes its wavelength as $\lambda$; the notebook uses $\lambda_0$ here to distinguish vacuum from in-medium wavelength.

<a id="ref-15"></a>

15. **Kirchauer (1998)**, [*Partial Coherence and Advanced Apertures*, §2.3.3](https://www.iue.tuwien.ac.at/phd/kirchauer/node21.html), doctoral dissertation, TU Wien, Eq. (2.4): illumination-to-projection NA ratio $\sigma$. Used here only for the source-size convention, with both NAs referred to the mask side.

## Selected freely accessible reading

Optional reading: start with the suggested passage. Freely accessible works are not necessarily in the public domain.

- **M. T. Tran and R. Oldenbourg. [An experimental method to characterize the relationship between aperture image and ray directions in microscope optics](https://pmc.ncbi.nlm.nih.gov/articles/PMC8005462/).** Microscopy Research and Technique 84, 668–674 (2021; online 2020); DOI: 10.1002/jemt.23625. **Access:** Free author manuscript in PMC; not public domain. **Read for:** Read Section 2.3 and Figure 2: calibrated angular-to-pupil mapping.

- **J. E. Harvey and R. N. Pfisterer. [Understanding diffraction grating behavior: including conical diffraction and Rayleigh anomalies from transmission gratings](https://www.spiedigitallibrary.org/journals/optical-engineering/volume-58/issue-08/087105/Understanding-diffraction-grating-behavior--including-conical-diffraction-and-Rayleigh/10.1117/1.OE.58.8.087105.full).** Optical Engineering 58, 087105 (2019); DOI: 10.1117/1.OE.58.8.087105. **Access:** Freely readable publisher full text; not public domain. **Read for:** Read Sections 2–4: signed directions and conical geometry.

**Convention check:** Translate each source's order signs, propagation side, polarization assumptions and amplitude/power normalization to this notebook's stated model.

[Source catalogue and access notes](reading_sources.json).


[Return to 5.4: Wavevectors, Sigma Coordinates and the Pupil](../5_4_From_Diffraction_Coordinates_to_Pupils.ipynb)